<a href="https://colab.research.google.com/github/openforest4d/lidar_basic_concepts_and_exercises/blob/main/notebooks/Act1_Act2_combined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Activity 1. Getting Started with lidar point clouds in R (Google Colab)

Authors: Madhumitha Katam and Chelsea Scott, Arizona State University.


Welcome! This notebook is your starting point for working with LiDAR data in R using Google Colab.

By the end of this activity 1 you will be able to:
- Set up R spatial packages in Colab
- Load a `.laz` point cloud file
- Read and understand its summary
- Visualize the point cloud in 2D

**No prior R experience needed.**

> **Note:** Run each cell in order. At the top left corner of each cell, you see a "play" button, click on it to run the cell.

## Step 1: Install R Packages

We need three packages:

| Package | Purpose |
|---|---|
| `lidR` | Read, process, and analyze LiDAR point clouds |
| `ggplot2` | Create 2D plots and maps |
| `plotly` | Create interactive 3D visualizations in the browser |



**Important Note**

1. Click on "Runtime" in the menu section on the top and select "Change Runtime Type"
2. A pop up menu apprears. Under "Runtime Type" section, click on the dropdown and select R.
3. Under "Runtime Session" section , click on the dropdown and select "2026.04" and click on Save.
4. Start running the below cells.

**PLEASE NOTE, THIS CELL TAKES 10-15 MIN TO RUN**





In [ ]:
# Option 1: Download compressed package containing all required libraries

system("wget https://zenodo.org/records/17611774/files/forestlib.tar.gz")
system("tar -xzf forestlib.tar.gz")

# Add forestlib to R library search path
.libPaths("forestlib")

# Install pacman only if not available
if (!requireNamespace("pacman", quietly = TRUE)) {
  install.packages("pacman")
}

# Load packages using pacman for easier management in Colab WITHOUT reinstalling
pacman::p_load(
  lidR,        # Lidar data processing
  terra,       # Raster handling
  raster,      # Legacy support (used in Compute Canopy Metrics Grid)
  sp,          # Spatial data structures
  sf,          # Simple features
  RCSF,         # Cloth Simulation Filter
  install = FALSE,
  update = FALSE
)
# pacman::p_load(
# lidR,# Lidar data processing
# sf, # Simple features
# install = TRUE)

## Step 2: Load Libraries

Now that the packages are installed, we load them into the current R session.

> **Note:** You need to run this every time you start a new Colab session, even if you already installed the packages.

In [ ]:
library(lidR)
library(ggplot2)
library(terra)

cat("All packages loaded successfully!\n")

## Step 3: Download the lidar Point Cloud (.laz) file




In [ ]:
# Pull the .laz file from internet.
laz_url_base <- "https://zenodo.org/records/21517293/files/"
zenodo_name  <- "ACT_UGS_Wasatch.laz"
laz_path     <- "ACT_UGS_Wasatch.laz"      # local filename
laz_url      <- paste0(laz_url_base, zenodo_name, "?download=1")

# Download the file to the Colab working directory
download.file(laz_url, destfile = laz_path, mode = "wb")
cat("Download complete:", laz_path, "\n")

## Step 4: Read the Lidar Point Cloud (.laz)

`readLAS()` loads the `.laz` file into R as a `LAS` object. This is the standard format used by the `lidR` package.

In [ ]:
las <- readLAS(laz_path)

# Filter out the noisy points
n_before <- npoints(las)
las <- filter_poi(las, !Classification %in% c(7, 18))

# Print a summary of the point cloud
# This shows: number of points, bounding box (X/Y/Z range),


cat("Type of file     :", class(las)[1], "\n")
cat("Coordinate extent:", paste(c(st_bbox(las)), collapse = ", "), "(xmin, ymin, xmax, ymax)\n")
cat("Area             :", round(area(las)), "m²\n")
cat("Points           :", npoints(las), "\n")
cat("Density          :", round(density(las), 2), "points/m²\n")

| Attribute | Description |
|-----------|-------------|
| Type of file | R object type. `LAS` is a point cloud object from the `lidR` package. |
| Coordinate extent | Bounding box of the point cloud (xmin, ymin, xmax, ymax) in CRS units (meters). |
| Area | Ground area covered by the point cloud, in square meters. |
| Points | Total number of LiDAR returns in the file. |
| Density | Average points per square meter (points / area). |

## Step 5: Explore the Lidar Point Cloud (.laz)

Before visualizing, it helps to look at the raw numbers like how many points, what elevation range, and what attributes are recorded for each point.


In [ ]:
# Total number of points
cat("Total points:", nrow(las@data), "\n\n")

# List all recorded attributes (columns) per point
cat("Available attributes:\n")
print(names(las@data))

In [ ]:
# Summary of Z (elevation/height) values
z_summary <- summary(las@data$Z)

cat("\nZ (elevation) summary:\n")
cat("Minimum elevation      =", z_summary["Min."], "m\n")
cat("1st quartile elevation =", z_summary["1st Qu."], "m\n")
cat("Median elevation       =", z_summary["Median"], "m\n")
cat("Mean elevation         =", z_summary["Mean"], "m\n")
cat("3rd quartile elevation =", z_summary["3rd Qu."], "m\n")
cat("Maximum elevation      =", z_summary["Max."], "m\n")

In [ ]:
# Classification breakdown using ASPRS LAS standard class codes
class_labels <- c(
  "0"  = "Never classified",
  "1"  = "Unassigned",
  "2"  = "Ground",
  "3"  = "Low vegetation",
  "4"  = "Medium vegetation",
  "5"  = "High vegetation",
  "6"  = "Building",
  "7"  = "Low point (noise)",
  "9"  = "Water",
  "10" = "Rail",
  "11" = "Road surface",
  "13" = "Wire - guard",
  "14" = "Wire - conductor",
  "15" = "Transmission tower",
  "17" = "Bridge deck",
  "18" = "High noise"
)

class_counts <- table(las@data$Classification)

cat("\nPoint classification counts:\n")
for (code in names(class_counts)) {
  label <- class_labels[code]
  if (is.na(label)) label <- "Unknown / user-defined"
  cat(sprintf("Class %s (%s) = %d points\n", code, label, class_counts[code]))
}

## Step 6: Plot the 2D Top-Down View of Lidar Point Cloud (.laz)
We use `ggplot2` to create a 2D bird's-eye view of the point cloud, colored by elevation. This is a quick way to see the spatial extent and structure of your data.

In [ ]:
# Convert the LAS object to a regular R dataframe for plotting
las_df <- as.data.frame(las@data)

options(repr.plot.width = 9, repr.plot.height = 7)

ggplot(las_df, aes(x = X, y = Y, color = Z)) +
  geom_point(size = 0.2, alpha = 0.4) +
  scale_color_viridis_c(name = "Elevation (m)") +
  coord_equal() +
  labs(
    title = "Point Cloud: Top-Down View",
    x = "Easting (m)",
    y = "Northing (m)"
  ) +
  theme_minimal(base_size = 12)

## Step 7: Plot the Lidar Point Cloud (.laz) by Intensity (Optional)

If your point cloud includes an `Intensity` attribute, you can color by that instead of elevation. Intensity reflects how strongly each surface reflected the laser pulse. It useful for distinguishing surface types like pavement, vegetation, or water.

In [ ]:
if ("Intensity" %in% names(las_df)) {

  options(repr.plot.width = 9, repr.plot.height = 7)

  ggplot(las_df, aes(x = X, y = Y, color = Intensity)) +
    geom_point(size = 0.2, alpha = 0.4) +
    scale_color_viridis_c(option = "inferno", name = "Intensity") +
    coord_equal() +
    labs(
      title = "Point Cloud: Colored by Intensity",
      subtitle = "Higher intensity = stronger laser return",
      x = "Easting (m)",
      y = "Northing (m)"
    ) +
    theme_minimal(base_size = 12)

} else {
  cat("Intensity attribute not found in this point cloud.\n")
}

## You're Ready!

You have successfully:
- Installed all required packages
- Loaded a LiDAR `.laz` file
- Read its summary and attributes
- Visualized it in 2D and 3D

**Next step:** Now go back to your document and answer the questions.


### ONLY RUN THIS IF YOU ARE CURRENTLY IN ACTIVITY 2

# Activity 2: Generating DEM, DTM and DSMs from Lidar Point Cloud (.laz)


By the end of this activity 2, you will

- understand the differences between DEM, DSM and DTM in detail
- Visualize all the rasters in hill shade view
- Create a canopy height model using DSM and DTM

## Step0 : Upload the Point Cloud File
Before loading your point cloud file, you need to upload it to Colab:

1. Click the **folder icon** in the left sidebar
2. Click the **upload icon** (arrow pointing up)
3. Select your `.laz` or `.las` file
4. Wait for the upload to complete — the file will appear under `/content/`
5. Uncomment the code. and run the cell.

Then update the filename `las_file` in the cell below to match your file.

In [ ]:
#Change the filename below to match your uploaded file
laz_path<- "/content/points.laz"

# Check the file exists before loading
if (file.exists(laz_path)) {
  cat("File found! Ready to load.\n")
} else {
  cat("File not found. Check the filename and make sure it is uploaded.\n")
}

# Read the laz file

las <- readLAS(laz_path)

# Filter out the noisy points
n_before <- npoints(las)
las <- filter_poi(las, !Classification %in% c(7, 18))

# Print a summary of the point cloud
# This shows: number of points, bounding box (X/Y/Z range),


cat("Type of file     :", class(las)[1], "\n")
cat("Coordinate extent:", paste(c(st_bbox(las)), collapse = ", "), "(xmin, ymin, xmax, ymax)\n")
cat("Area             :", round(area(las)), "m²\n")
cat("Points           :", npoints(las), "\n")
cat("Density          :", round(density(las), 2), "points/m²\n")

## Step1 : Digital Elevation Model (DEM)

A Digital Elevation Model (DEM) is a raster representation of elevation derived from the point cloud. Here, for teaching purposes, we compute it as the **mean Z of ALL returns** in each 1-meter cell (ground, vegetation, and buildings all mixed together), with no filtering by classification.

In [ ]:
# rasterize_density for point density map; swap below for elevation
# For a NAIVE elevation raster (mean Z of ALL points, no class filter):
# NOTE: this is not a true bare-earth DEM -- see markdown above.
dem <- pixel_metrics(las, fun = ~mean(Z), res = 1)

cat("Naive elevation raster generated (mean Z of all points). Resolution: ", res(dem)[1], "m x ", res(dem)[2], "m\n", sep = "")
cat("Reminder: this mixes ground + vegetation/building returns; compare with the DTM in Step 3.\n")

## Plot the DEM
All the plots of the rasters are hillshade views. A hillshade is a nice way to visualize data. It shows the landscape as if it were lit up by the sun from a particular direction, so slopes facing the light look bright and slopes facing away look dark. That mix of light and shadow gives a flat elevation map a sense of depth, making hills, valleys, and ridges much easier to see.

In [ ]:
# Convert SpatRaster 'dem' to RasterLayer for raster::terrain
dem_raster <- as(dem, "Raster")

# Calculate slope and aspect using the 'raster' package's terrain function
slope_aspect_stack <- raster::terrain(dem_raster, opt = c('slope', 'aspect'))

# Extract slope and aspect layers and convert to SpatRaster for terra::shade
slope_raster <- terra::rast(slope_aspect_stack[[1]])
aspect_raster <- terra::rast(slope_aspect_stack[[2]])

# Generate hillshade using the calculated slope and aspect
hillshade <- terra::shade(slope_raster, aspect_raster, angle = 45, direction = 270)

# Plot the hillshade
plot(hillshade,
     main = "Hillshade from DEM",
     col = grey(0:100/100),
     legend = FALSE,
     xlab = "Easting (m)",
     ylab = "Northing (m)")

## Step 2: Digital Surface Model (DSM)
A Digital Surface Model (DSM) is a raster representation of the **topmost surface** of the landscape, including vegetation, buildings, and other above-ground features. It is created by taking the highest LiDAR return within each grid cell.

In [ ]:
# rasterize_canopy creates a Digital Surface Model (DSM) by taking the highest point (Z value)
# within each grid cell. This captures the top surface of objects like trees and buildings.
dsm <- rasterize_canopy(las, res = 1, algorithm = p2r()) # p2r() algorithm selects the highest return per cell

# Print a message confirming the DSM generation and its resolution
cat("DSM generated. Resolution: ", res(dsm)[1], "m x ", res(dsm)[2], "m\n", sep = "")

## Plot the DSM



In [ ]:
# Convert SpatRaster 'dsm' to RasterLayer for raster::terrain
dsm_raster <- as(dsm, "Raster")

# Calculate slope and aspect using the 'raster' package's terrain function
slope_aspect_stack <- raster::terrain(dsm_raster, opt = c('slope', 'aspect'))

# Extract slope and aspect layers and convert to SpatRaster for terra::shade
slope_raster <- terra::rast(slope_aspect_stack[[1]])
aspect_raster <- terra::rast(slope_aspect_stack[[2]])

# Generate hillshade using the calculated slope and aspect
hillshade <- terra::shade(slope_raster, aspect_raster, angle = 45, direction = 270)

# Plot the hillshade
plot(hillshade, main = "Hillshade from DSM", col = grey(0:100/100), legend = FALSE, xlab = "Easting (m)",
     ylab = "Northing (m)")

## Step 3: Digital Terrain Model (DTM)
A Digital Terrain Model (DTM) is a raster representation of the bare ground surface, with vegetation and buildings removed. It is created by interpolating ground-classified LiDAR points into a continuous elevation grid. The DTM serves as the reference surface for normalizing point heights in later steps.


### Check for ground points before generating the DTM


In [ ]:
# Sanity check: rasterize_terrain() needs ground-classified points (Class 2)
# or water points (Class 9) to interpolate a terrain surface. If your uploaded
# file lacks these, the DTM step below will fail or produce a poor/extrapolated surface.
ground_pts <- sum(las@data$Classification %in% c(2, 9))
cat("Ground/water points available for DTM:", ground_pts,
    sprintf("(%.1f%% of total)\n", 100 * ground_pts / npoints(las)))

if (ground_pts == 0) {
  cat("WARNING: No ground/water-classified points found. rasterize_terrain() will fail.\n",
      "You may need a differently-classified point cloud, or classify ground points first\n",
      "(e.g. with classify_ground()) before running the DTM step below.\n")
}

In [ ]:
# rasterize_terrain uses ground (Class 2) and water (Class 9) points internally by default (use_class = c(2, 9))
# Alternative: use tin() for a TIN-based interpolation
dtm <- rasterize_terrain(las, res = 1, algorithm = tin())

cat("DTM generated. Resolution: ", res(dtm)[1], "m x ", res(dtm)[2], "m\n", sep = "")

## Plot the DTM

In [ ]:
# Convert SpatRaster 'dtm' to RasterLayer for raster::terrain
dtm_raster <- as(dtm, "Raster")

# Calculate slope and aspect using the 'raster' package's terrain function
slope_aspect_stack <- raster::terrain(dtm_raster, opt = c('slope', 'aspect'))

# Extract slope and aspect layers and convert to SpatRaster for terra::shade
slope_raster <- terra::rast(slope_aspect_stack[[1]])
aspect_raster <- terra::rast(slope_aspect_stack[[2]])

# Generate hillshade using the calculated slope and aspect
hillshade <- terra::shade(slope_raster, aspect_raster, angle = 45, direction = 270)

# Plot the hillshade
plot(hillshade, main = "Hillshade from DTM", col = grey(0:100/100), legend = FALSE, xlab = "Easting (m)",
     ylab = "Northing (m)")

## Step 4: Canopy Height Model (CHM)

The Canopy Height Model (CHM) measures vegetation height in natural areas and building height in urban areas. It is calculated through two methods: the difference between DSM and DTM (as coded below) or height normalization.


In [ ]:
chm <- dsm - dtm
chm[chm < 0] <- 0
cat("CHM generated. Resolution: ", res(chm)[1], "m x ", res(chm)[2], "m\n", sep = "")

### Characteristics of Canopy Height Model

In [ ]:
min_chm <- global(chm, fun = "min", na.rm = TRUE)$min
max_chm <- global(chm, fun = "max", na.rm = TRUE)$max
mean_chm <- global(chm, fun = "mean", na.rm = TRUE)$mean
sd_chm <- global(chm, fun = "sd", na.rm = TRUE)$sd

cat("CHM Z-Statistic Values:\n")
cat(paste0("  Minimum Height    : ", round(min_chm, 2), " meters\n"))
cat(paste0("  Maximum Height.   : ", round(max_chm, 2), " meters\n"))
cat(paste0("  Mean Height.      : ", round(mean_chm, 2), " meters\n"))
cat(paste0("  Standard Deviation: ", round(sd_chm, 2), " meters\n"))

## Plot the CHM

In [ ]:
# Plot CHM with white colors representing 0 and green representing tall trees
plot(chm, main = "Canopy Height Model (CHM)", col = rev(hcl.colors(100, "Greens")), xlab = "Easting (m)",
     ylab = "Northing (m)")

In [ ]:
# Generate a histogram of CHM values
hist(chm, main = "Histogram of CHM Values", xlab = "Canopy Height (m)", col = hcl.colors(1, "Greens"), breaks = 25)